In [ ]:
# https://www.b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/market-data/historico/mercado-a-vista/series-historicas/

In [ ]:
import sys
import os
sys.path
os.listdir()
os.chdir(os.getcwd().replace("\\","/").replace("/notebooks",""))
sys.path.append("src")

In [ ]:
from src.envConfig import EnvConfig

In [ ]:
EnvConfig()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, substring, trim, expr

In [ ]:
# 1. Inicializa a sessão Spark
spark = SparkSession.builder \
    .appName("Normalizador_COTAHIST_B3") \
    .getOrCreate()

In [ ]:
# 2. Carrega o arquivo txt como texto bruto (uma coluna única chamada 'value')
caminho_arquivo = ""

In [ ]:
df_raw = spark.read.text(caminho_arquivo)

In [ ]:

# 3. Filtra apenas o Tipo de Registro "01" (Cotações por papel)
# Isso ignora o Header (00) e o Trailer (99)
df_cotacoes = df_raw.filter(substring(col("value"), 1, 2) == "01")

# 4. Aplica o fatiamento (substring) baseado no layout oficial da B3
df_normalizado = df_cotacoes.select(
    # Data: posições 3 a 10 (AAAA-MM-DD)
    expr("to_date(substring(value, 3, 8), 'yyyyMMdd')").alias("data_pregao"),
    
    # Código BDI: posições 11 a 12
    trim(substring(col("value"), 11, 2)).alias("cod_bdi"),
    
    # Ticker: posições 13 a 24
    trim(substring(col("value"), 13, 12)).alias("ticker"),
    
    # Tipo de Mercado: posições 25 a 27
    trim(substring(col("value"), 25, 3)).alias("tipo_mercado"),
    
    # Nome da Empresa: posições 28 a 39
    trim(substring(col("value"), 28, 12)).alias("nome_empresa"),
    
    # Especificação do Papel (Ex: ON, PN): posições 40 a 49
    trim(substring(col("value"), 40, 10)).alias("especificacao_papel"),
    
    # Prazo em dias (Mercado a Termo): posições 50 a 52
    trim(substring(col("value"), 50, 3)).alias("prazo_termo"),
    
    # Moeda: posições 53 a 56
    trim(substring(col("value"), 53, 4)).alias("moeda_referencia"),
    
    # Preços (posições fixas convertidas para Decimal e divididas por 100 para ajustar os centavos)
    (substring(col("value"), 57, 13).cast("decimal(13,2)") / 100).alias("preco_abertura"),
    (substring(col("value"), 70, 13).cast("decimal(13,2)") / 100).alias("preco_maximo"),
    (substring(col("value"), 83, 13).cast("decimal(13,2)") / 100).alias("preco_minimo"),
    (substring(col("value"), 96, 13).cast("decimal(13,2)") / 100).alias("preco_medio"),
    (substring(col("value"), 109, 13).cast("decimal(13,2)") / 100).alias("preco_fechamento"),
    (substring(col("value"), 122, 13).cast("decimal(13,2)") / 100).alias("preco_melhor_oferta_compra"),
    (substring(col("value"), 135, 13).cast("decimal(13,2)") / 100).alias("preco_melhor_oferta_venda"),
    
    # Negócios e Volumes
    substring(col("value"), 148, 5).cast("int").alias("total_negocios"),
    substring(col("value"), 153, 18).cast("bigint").alias("quantidade_titulos"),
    (substring(col("value"), 171, 18).cast("decimal(18,2)") / 100).alias("volume_total"),
    
    # Código ISIN: posições 231 a 242
    trim(substring(col("value"), 231, 12)).alias("codigo_isin")
)



In [ ]:
# 5. Exibe uma prévia dos dados processados
df_normalizado.show(5, truncate=False)


In [ ]:
df_normalizado.filter(col("ticker") == "SNEL11").show()

In [ ]:
# 6. Salva o resultado final em um formato performático (Parquet ou Delta)
# df_normalizado.write.mode("overwrite").parquet("output/b3_cotahist_normalizado")